# `02_all_variants_new_opt_67b.ipynb` — NEW-feature variants on `facebook/opt-6.7b`

This notebook tests **4 new variants (M, N, O, P)** built on **6 new features (F11-F16)** proposed after the OPT-6.7B baseline run showed all 12 existing variants underperforming HalluShift (avg AUROC 0.536) and EigenScore (0.509).

## New features

| Feature | Name | Dim | Paper inspiration |
|---|---|---|---|
| **F11** | Unembedding-Reasoning Projection (URP) | 8 | HARP (arXiv 2509.11536, Sep 2025) |
| **F12** | Layer-Trajectory Curvature (LTC) | 2 | LSD / Geometry of Truth (arXiv 2510.04933, Oct 2025) |
| **F13** | Confidence-Trajectory Slope (CTS) | 2 | ShED-HD (arXiv 2503.18242) |
| **F14** | Effective Attention Rank (EAR) | 1 | Spectral attention features (arXiv 2502.17598) |
| **F15** | Prompt-Echo Alignment (PEA) | 1 | Original — hidden-state alignment vs F1's attention-mass |
| **F16** | Head-Importance Divergence (HID) | 1 | Original — higher-moment statistic on per-head concentration |

Total new feature dimension: **15 scalars** (URP 8 + LTC 2 + CTS 2 + EAR + PEA + HID).

## New variants

| Variant | Composition | Hypothesis |
|---|---|---|
| **M** | canonical + URP + LTC | Geometry-focused (output projection + curvature) |
| **N** | canonical + CTS + EAR + PEA | Trajectory+attention focused |
| **O** | canonical + URP + LTC + CTS | Best-of-new trio |
| **P** | canonical + D\_mean + V\_last + H\_mean + URP + LTC + CTS + EAR + PEA + HID | Everything-new on top of MIND+ |

## Comparison target

The existing OPT-6.7B baseline results:
- HalluShift: avg AUROC **0.536** (current ceiling)
- EigenScore: avg AUROC 0.509
- SAPLMA: avg AUROC 0.502
- Best existing variant (J): avg AUROC 0.436

**Goal of this notebook:** test whether any of M / N / O / P closes the gap to HalluShift (and ideally exceeds it).

## Outputs

- `kaggle_opt_67b_all_variants_NEW_results.json` — full metrics dump
- `kaggle_opt_67b_variant_M_best.pth` through `_P_best.pth` — trained MLPs
- `kaggle_opt_67b_features_NEW.json` — cached feature vectors (skip-on-rerun)

Total budget: ~80-100 min on Kaggle T4×2 (1000 records/class, eager attention, single forward pass per record for feature extraction; sub-minute MLP training per variant).


In [ ]:
# =============================================================================
# BLOCK 0: PIP INSTALLS
# =============================================================================
!pip install -q -U "transformers>=4.45" "tokenizers>=0.19" "accelerate>=0.30"
!pip install -q -U datasets nltk scikit-learn tqdm sentence-transformers scipy pyarrow


In [ ]:
# =============================================================================
# BLOCK 0.5: KAGGLE INPUT SHIM — copy uploaded datasets into the working dir
# =============================================================================
import os, glob, shutil
MODEL_TAG_LOCAL = "kaggle_opt_67b"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(WORK, exist_ok=True)

if os.path.isdir("/kaggle/input"):
    print("Attached Kaggle datasets:")
    for d in sorted(os.listdir("/kaggle/input")):
        print(f"  /kaggle/input/{d}")
        for f in sorted(os.listdir(f"/kaggle/input/{d}"))[:5]:
            print(f"     {f}")

    # dataset_full.json
    hits = glob.glob(f"/kaggle/input/**/{MODEL_TAG_LOCAL}_dataset_full.json", recursive=True)
    if hits:
        src = hits[0]
        dst = f"{WORK}/{MODEL_TAG_LOCAL}_dataset_full.json"
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        print(f"\n[OK] dataset_full: {src} -> {dst}  ({os.path.getsize(dst)/1e6:.1f} MB)")
    else:
        print(f"\n[WARN] {MODEL_TAG_LOCAL}_dataset_full.json not found under /kaggle/input/.")
        print("       Make sure you attached the dataset_full.json dataset.")

    # eval_*.parquet
    pq_hits = glob.glob("/kaggle/input/**/eval_*.parquet", recursive=True)
    for src in pq_hits:
        dst = f"{WORK}/{os.path.basename(src)}"
        if not os.path.exists(dst):
            shutil.copy(src, dst)
    print(f"[OK] copied {len(pq_hits)} eval_*.parquet files into {WORK}")


In [ ]:
# =============================================================================
# BLOCK 1 (STAGE 1): SETUP + LOAD dataset_full.json
# =============================================================================
import os, sys, gc, json, random, math, time, datetime, platform, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

MODEL_NAME       = "facebook/opt-6.7b"
MODEL_TAG        = "kaggle_opt_67b"
SEED             = 42
DTYPE            = torch.float16 if torch.cuda.is_available() else torch.float32
TEST_FRAC        = 0.20

# Per-model layer choices for OPT-6.7B (32 layers, hidden=4096)
SAPLMA_LAYER     = 20          # ~63% depth, paper-canonical for OPT-6.7B
URP_LAYER        = -1          # last hidden state for URP projection
LTC_USE_ALL      = True        # use all layers for trajectory curvature
CTS_NEEDS        = "logits"    # use forward-pass logits (no generation)
MID_LAYER        = 16          # middle layer for EAR, PEA, HID
URP_K            = 8           # number of bottom-k singular vectors for URP basis

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PATH_DATASET_FULL    = f"{MODEL_TAG}_dataset_full.json"
PATH_FEATURES_NEW    = f"{MODEL_TAG}_features_NEW.json"
PATH_RESULTS         = f"{MODEL_TAG}_all_variants_NEW_results.json"

DIAG = {
    "schema_version": "all-variants-NEW-1.0",
    "model_tag":  MODEL_TAG,
    "model_name": MODEL_NAME,
    "config": {
        "seed": SEED, "dtype": str(DTYPE), "test_frac": TEST_FRAC,
        "urp_layer": URP_LAYER, "urp_k": URP_K,
        "mid_layer": MID_LAYER,
        "feature_set": ["URP", "LTC", "CTS", "EAR", "PEA", "HID"],
        "variants": ["M", "N", "O", "P"],
    },
    "env": {
        "python":   sys.version.split()[0],
        "platform": platform.platform(),
        "torch":    torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device":   (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
        "free_vram_gb":  (round(torch.cuda.mem_get_info()[0]/1e9, 2) if torch.cuda.is_available() else None),
    },
    "library_versions": {},
    "stage_timings_sec": {},
    "variants": {},
    "downstream_datasets": {},
    "errors": [],
}
for _mod in ["transformers", "datasets", "accelerate", "sklearn", "scipy", "numpy", "pyarrow"]:
    try:
        m = __import__(_mod)
        DIAG["library_versions"][_mod] = getattr(m, "__version__", "n/a")
    except Exception as e:
        DIAG["library_versions"][_mod] = f"IMPORT_FAILED: {e}"

# Load dataset_full.json (produced by 01_data_generation_opt_67b.ipynb)
if not os.path.exists(PATH_DATASET_FULL):
    raise SystemExit(
        f"[ERROR] {PATH_DATASET_FULL} not found. Either:\n"
        f"  1. Run 01_data_generation_opt_67b.ipynb first (it produces this file), or\n"
        f"  2. Upload {PATH_DATASET_FULL} from your previous 01 run.")
with open(PATH_DATASET_FULL, "r") as f:
    records_full = json.load(f)
print(f"\u2713 loaded {len(records_full)} records from {PATH_DATASET_FULL}")
print(f"  label dist: y=0: {sum(1 for r in records_full if r['label']==0)}   "
      f"y=1: {sum(1 for r in records_full if r['label']==1)}")

# 80/20 split — SAME seed as old 02 so AUROCs comparable to existing variant results
random.seed(SEED)
records_shuf = list(records_full)
random.shuffle(records_shuf)
split_idx = int((1.0 - TEST_FRAC) * len(records_shuf))
train_recs = records_shuf[:split_idx]
test_recs  = records_shuf[split_idx:]
print(f"  train n = {len(train_recs)}   test n = {len(test_recs)}")
DIAG["n_train"] = len(train_recs); DIAG["n_test"] = len(test_recs)


In [ ]:
# =============================================================================
# BLOCK 2 (STAGE 2): LOAD MODEL (eager attention)  +  SVD for URP basis
# =============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM

_LLM = {"tokenizer": None, "model": None, "urp_basis": None}

def need_llm():
    if _LLM["model"] is not None:
        return _LLM
    print(f"Loading {MODEL_NAME} (dtype={DTYPE}) ...")
    _t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    load_kwargs = dict(torch_dtype=DTYPE,
                       device_map="auto" if torch.cuda.is_available() else None,
                       attn_implementation="eager")
    try:
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    except Exception as e:
        print(f"[warn] retry load without attn_implementation: {e}")
        load_kwargs.pop("attn_implementation", None)
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad = False
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    DIAG["stage_timings_sec"]["model_load"] = round(time.perf_counter() - _t0, 2)
    DIAG["model_info"] = {
        "name": MODEL_NAME,
        "hidden_dim": int(mdl.config.hidden_size),
        "n_layers": int(mdl.config.num_hidden_layers),
        "vocab_size": int(mdl.config.vocab_size),
    }
    print(f"\u2713 OPT-6.7B loaded in {DIAG['stage_timings_sec']['model_load']}s  "
          f"hidden_dim={mdl.config.hidden_size}  layers={mdl.config.num_hidden_layers}")

    # --- SVD on unembedding matrix for URP basis -----------------------------
    print("Computing URP basis (SVD on output embedding) ...")
    _t1 = time.perf_counter()
    with torch.no_grad():
        W_U = mdl.get_output_embeddings().weight.detach().float().cpu()  # [V, d]
        # SVD: W_U = U @ diag(S) @ Vh, Vh shape [d, d]
        # Reasoning subspace = bottom-k right singular vectors (smallest singular values)
        U_, S_, Vh = torch.linalg.svd(W_U, full_matrices=False)
        urp_basis = Vh[-URP_K:, :].contiguous()   # [k, d]
    _LLM["urp_basis"] = urp_basis.to(mdl.device, dtype=torch.float32)
    DIAG["stage_timings_sec"]["urp_svd"] = round(time.perf_counter() - _t1, 2)
    print(f"  URP basis shape {urp_basis.shape}  (took {DIAG['stage_timings_sec']['urp_svd']}s)")

    _LLM["tokenizer"] = tok; _LLM["model"] = mdl
    return _LLM


In [ ]:
# =============================================================================
# BLOCK 3a (STAGE 3a): FEATURE EXTRACTORS — F11-F16 + canonical + MIND+ scalars
# =============================================================================
def _safe_cos(a, b, eps=1e-8):
    na = a.norm() + eps; nb = b.norm() + eps
    return float((a @ b) / (na * nb))


def _renyi2_concentration(p):
    """Renyi-2 entropy of distribution p: -log sum(p^2). High = uniform, low = concentrated."""
    p2 = (p * p).sum().clamp_min(1e-12)
    return float(-torch.log(p2))


@torch.no_grad()
def extract_all_new_features(text):
    """One forward pass returning a dict of all new + canonical + MIND+ features.

    Returns:
      embedding   : list[float]  (last-token last-layer hidden)
      D_mean      : float        (canonical MIND+ #1)
      V_last      : float        (canonical MIND+ #2)
      H_mean      : float        (canonical MIND+ #3)
      F11_urp     : list[float]  (URP_K=8 scalars)
      F12_ltc_mean: float
      F12_ltc_max : float
      F13_cts_slope    : float
      F13_cts_variance : float
      F14_ear     : float
      F15_pea     : float
      F16_hid     : float
    """
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    URP_BASIS = LL["urp_basis"]   # [k, d]

    enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                    max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
    input_ids = enc.input_ids
    T = int(input_ids.shape[1])

    out = model(**enc, output_hidden_states=True, output_attentions=True, use_cache=False)
    hidden_states = out.hidden_states     # tuple of (L+1) tensors, each [1, T, d]
    attentions    = out.attentions         # tuple of L tensors, each [1, n_heads, T, T]
    logits        = out.logits             # [1, T, V]
    L = len(hidden_states) - 1             # number of transformer layers

    # ---------------- Canonical (MIND backbone): last-token last-layer hidden -
    h_last_top = hidden_states[-1][0, -1, :].float()   # [d]
    embedding = h_last_top.cpu().numpy().astype(np.float32).tolist()

    # ---------------- MIND+ scalars (D_mean, V_last, H_mean) -----------------
    # D_mean: average cosine distance between adjacent layers at last token
    last_token_per_layer = torch.stack([hidden_states[l+1][0, -1, :].float() for l in range(L)], dim=0)   # [L, d]
    D_arr = []
    for l in range(L - 1):
        D_arr.append(1.0 - _safe_cos(last_token_per_layer[l], last_token_per_layer[l+1]))
    D_mean = float(np.mean(D_arr)) if D_arr else 0.0

    # V_last: L2 variance of last-token activations across layers
    H_mean_layer = last_token_per_layer.mean(dim=0)    # [d]
    V_last = float(((last_token_per_layer - H_mean_layer) ** 2).sum(dim=1).mean())

    # H_mean: mean per-step Shannon entropy of the output token distribution
    probs = F.softmax(logits[0].float(), dim=-1).clamp_min(1e-12)   # [T, V]
    per_step_H = -(probs * torch.log(probs)).sum(dim=-1)            # [T]
    H_mean = float(per_step_H.mean())

    # ---------------- F11: Unembedding-Reasoning Projection (URP) ------------
    # Project last-token last-layer hidden onto bottom-k right singular vectors of W_U.
    f11_urp = (URP_BASIS.to(h_last_top.dtype) @ h_last_top).cpu().numpy().astype(np.float32).tolist()

    # ---------------- F12: Layer-Trajectory Curvature (LTC) ------------------
    # Direction change between adjacent layer transitions.
    deltas = last_token_per_layer[1:] - last_token_per_layer[:-1]   # [L-1, d]
    curvature = []
    for l in range(len(deltas) - 1):
        curvature.append(1.0 - _safe_cos(deltas[l], deltas[l+1]))
    if curvature:
        F12_ltc_mean = float(np.mean(curvature))
        F12_ltc_max  = float(np.max(curvature))
    else:
        F12_ltc_mean = 0.0
        F12_ltc_max  = 0.0

    # ---------------- F13: Confidence-Trajectory Slope (CTS) -----------------
    # Per-position top-1 log-probability trajectory.
    # Use logits at positions [1..T-1] predicting tokens [1..T-1] (one-step ahead).
    if T >= 4:
        p_max_per_t = probs.max(dim=-1).values   # [T]
        log_pmax = torch.log(p_max_per_t.clamp_min(1e-12)).cpu().numpy().astype(np.float64)
        # slope by linear regression
        x = np.arange(len(log_pmax), dtype=np.float64)
        try:
            slope, _ = np.polyfit(x, log_pmax, 1)
        except Exception:
            slope = 0.0
        F13_cts_slope    = float(slope)
        F13_cts_variance = float(np.var(log_pmax))
    else:
        F13_cts_slope    = 0.0
        F13_cts_variance = 0.0

    # ---------------- F14: Effective Attention Rank (EAR) --------------------
    # Spectral entropy of head-averaged attention matrix at mid layer.
    mid_attn = attentions[MID_LAYER][0].float()                # [n_heads, T, T]
    head_avg = mid_attn.mean(dim=0)                              # [T, T]
    try:
        sv = torch.linalg.svdvals(head_avg)
        p_sv = sv / sv.sum().clamp_min(1e-12)
        ear_H = -(p_sv * torch.log(p_sv.clamp_min(1e-12))).sum()
        F14_ear = float(torch.exp(ear_H))
    except Exception:
        F14_ear = 0.0

    # ---------------- F15: Prompt-Echo Alignment (PEA) -----------------------
    # Cosine between mean prompt hidden and mean generated hidden at mid layer.
    idx = text.find(". ")
    if idx == -1: idx = len(text) // 2
    p_enc = tokenizer(text[:idx + 2 if idx != -1 else len(text)//2].strip(),
                       return_tensors="pt", truncation=True,
                       max_length=getattr(model.config, "max_position_embeddings", 4096)).input_ids
    prompt_len = max(1, min(p_enc.shape[1], T - 1))
    mid_hidden = hidden_states[MID_LAYER + 1][0].float()          # [T, d]  (+1: layer 0 is embeddings)
    if prompt_len < T:
        mu_prompt = mid_hidden[:prompt_len].mean(dim=0)
        mu_gen    = mid_hidden[prompt_len:].mean(dim=0)
        F15_pea = _safe_cos(mu_prompt, mu_gen)
    else:
        F15_pea = 1.0

    # ---------------- F16: Head-Importance Divergence (HID) ------------------
    # Coefficient of variation of per-head Renyi-2 concentration at mid layer.
    head_concentrations = []
    for h in range(mid_attn.shape[0]):
        p_h = mid_attn[h].mean(dim=0)                            # [T] — avg attention over query positions
        p_h = p_h / p_h.sum().clamp_min(1e-12)
        head_concentrations.append(_renyi2_concentration(p_h))
    if head_concentrations:
        head_concentrations = np.array(head_concentrations, dtype=np.float64)
        mu = float(np.mean(head_concentrations))
        sigma = float(np.std(head_concentrations))
        F16_hid = float(sigma / max(mu, 1e-12))
    else:
        F16_hid = 0.0

    return {
        "embedding":         embedding,
        "D_mean":            D_mean,
        "V_last":            V_last,
        "H_mean":            H_mean,
        "F11_urp":           f11_urp,           # list of 8 floats
        "F12_ltc_mean":      F12_ltc_mean,
        "F12_ltc_max":       F12_ltc_max,
        "F13_cts_slope":     F13_cts_slope,
        "F13_cts_variance":  F13_cts_variance,
        "F14_ear":           F14_ear,
        "F15_pea":           F15_pea,
        "F16_hid":           F16_hid,
    }


# Keys for the variant feature stack (scalar features that get standardised)
NEW_SCALAR_KEYS = [
    "F12_ltc_mean", "F12_ltc_max",
    "F13_cts_slope", "F13_cts_variance",
    "F14_ear", "F15_pea", "F16_hid",
]
# URP is a vector — handled separately because each element is a scalar
NEW_URP_KEY = "F11_urp"   # list of 8 floats
URP_SUBKEYS = [f"F11_urp_{i}" for i in range(URP_K)]
print(f"\u2713 feature extractor defined. URP_K={URP_K}  scalar keys: {NEW_SCALAR_KEYS}")


In [ ]:
# =============================================================================
# BLOCK 3b (STAGE 3b): EXTRACT NEW FEATURES FOR ALL RECORDS (cache + fail-fast)
# =============================================================================
from tqdm.auto import tqdm

if os.path.exists(PATH_FEATURES_NEW):
    print(f"[SKIP STAGE 3] {PATH_FEATURES_NEW} exists — loading.")
    with open(PATH_FEATURES_NEW, "r") as _f:
        records_full = json.load(_f)
    print(f"  loaded {len(records_full)} feature vectors")
else:
    print(f"Extracting NEW features for {len(records_full)} records ...")
    need_llm()
    _t0 = time.perf_counter()

    # Fail-fast guard: abort after 10 consecutive failures (preserves transient
    # OOM tolerance for isolated bad rows, but aborts within ~30 seconds on
    # systematic NameError / missing import problems).
    FAILFAST_MAX_CONSECUTIVE = 10
    consecutive_failures = 0
    first_failure_exc = None
    failures = 0

    for i, r in enumerate(tqdm(records_full, desc="extract_new_features")):
        try:
            feats = extract_all_new_features(r["text"])
            r.update(feats)
            # Also flatten URP into URP_K scalar keys so build_X_y can pull them
            for j, v in enumerate(feats["F11_urp"]):
                r[f"F11_urp_{j}"] = float(v)
            consecutive_failures = 0
        except Exception as e:
            failures += 1
            consecutive_failures += 1
            if failures == 1:
                first_failure_exc = e
                print(f"\n[!!] FIRST FAILURE at row {i}:")
                print(f"     {type(e).__name__}: {e}")
                traceback.print_exc()
                print()
            DIAG["errors"].append(f"extract_new_features row {i}: {type(e).__name__}: {e}")
            # pad with zeros
            r["embedding"]        = [0.0] * DIAG["model_info"]["hidden_dim"]
            r["D_mean"]           = 0.0
            r["V_last"]           = 0.0
            r["H_mean"]           = 0.0
            r["F11_urp"]          = [0.0] * URP_K
            for j in range(URP_K): r[f"F11_urp_{j}"] = 0.0
            r["F12_ltc_mean"]     = 0.0
            r["F12_ltc_max"]      = 0.0
            r["F13_cts_slope"]    = 0.0
            r["F13_cts_variance"] = 0.0
            r["F14_ear"]          = 0.0
            r["F15_pea"]          = 0.0
            r["F16_hid"]          = 0.0
            if consecutive_failures >= FAILFAST_MAX_CONSECUTIVE:
                err_type = type(first_failure_exc).__name__
                raise RuntimeError(
                    f"\n\n[FAIL-FAST] extract_new_features hit {consecutive_failures} consecutive failures "
                    f"(first was at row {i - consecutive_failures + 1}, {err_type}: {first_failure_exc}). "
                    f"This looks systematic, not transient. Common causes:\n"
                    f"  - stale notebook (re-download from GitHub)\n"
                    f"  - missing pip package (re-run BLOCK 0)\n"
                    f"  - mismatched Python/transformers version\n"
                    f"Aborting now instead of producing 2000 zero-feature rows."
                )

    DIAG["stage_timings_sec"]["feature_extraction"] = round(time.perf_counter() - _t0, 2)
    DIAG["n_feature_failures"] = failures

    # Quick statistics for sanity
    DIAG["feature_stats"] = {}
    for k in NEW_SCALAR_KEYS + URP_SUBKEYS:
        vals = np.array([r[k] for r in records_full], dtype=np.float64)
        DIAG["feature_stats"][k] = {
            "mean": float(np.nanmean(vals)),
            "std":  float(np.nanstd(vals)),
            "min":  float(np.nanmin(vals)),
            "max":  float(np.nanmax(vals)),
        }

    with open(PATH_FEATURES_NEW, "w") as _f:
        json.dump(records_full, _f)
    print(f"\n\u2713 wrote {PATH_FEATURES_NEW}  ({os.path.getsize(PATH_FEATURES_NEW)/1e6:.2f} MB)")
    print(f"  feature_extraction: {DIAG['stage_timings_sec']['feature_extraction']} s   failures: {failures}")
    print("  Per-feature stats:")
    for k in NEW_SCALAR_KEYS:
        s = DIAG["feature_stats"][k]
        print(f"    {k:25s} mean={s['mean']:8.4f}  std={s['std']:8.4f}")

if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()


In [ ]:
# =============================================================================
# BLOCK 4 (STAGE 4): TRAIN 4 NEW MLP VARIANTS + WIKIPEDIA HELD-OUT EVAL
# =============================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              roc_auc_score, confusion_matrix, brier_score_loss)

# All scalar keys that variants can pull from
NEW_SCALAR_KEYS_FULL = NEW_SCALAR_KEYS + URP_SUBKEYS   # 7 + 8 = 15

VARIANTS = {
    # M: canonical + URP + LTC (geometry-focused)
    "M": URP_SUBKEYS + ["F12_ltc_mean", "F12_ltc_max"],

    # N: canonical + CTS + EAR + PEA (trajectory + attention)
    "N": ["F13_cts_slope", "F13_cts_variance", "F14_ear", "F15_pea"],

    # O: canonical + URP + LTC + CTS (best-of-new trio)
    "O": URP_SUBKEYS + ["F12_ltc_mean", "F12_ltc_max",
                         "F13_cts_slope", "F13_cts_variance"],

    # P: canonical + MIND+ stack + ALL new features (most ambitious)
    "P": ["D_mean", "V_last", "H_mean"] + URP_SUBKEYS + [
         "F12_ltc_mean", "F12_ltc_max",
         "F13_cts_slope", "F13_cts_variance",
         "F14_ear", "F15_pea", "F16_hid"],
}

print(f"Variants to train: {list(VARIANTS.keys())}")
for k, ks in VARIANTS.items():
    print(f"  {k}: {len(ks)} scalar features")

# In-memory MLPs + scalers for Stage 6 (multi-task eval)
TRAINED = {}

# 80/20 split with seed=42 — IDENTICAL across all variants
random.seed(SEED)
records_shuf = list(records_full)
random.shuffle(records_shuf)
split_idx = int((1.0 - TEST_FRAC) * len(records_shuf))
train_recs = records_shuf[:split_idx]
test_recs  = records_shuf[split_idx:]
print(f"Split: train {len(train_recs)}   test {len(test_recs)}")
hidden_dim = len(records_full[0]["embedding"])


class MINDPlusClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x): return self.layers(x)


class _FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def _nan_fill(arr):
    if not np.isnan(arr).any(): return arr
    col_means = np.nanmean(arr, axis=0)
    inds = np.where(np.isnan(arr))
    arr[inds] = np.take(col_means, inds[1])
    return arr


def build_X_y(recs, keys, scaler=None, fit=False):
    canon = np.array([r["embedding"] for r in recs], dtype=np.float32)
    if keys:
        scalars = np.array([[r.get(k, 0.0) for k in keys] for r in recs], dtype=np.float32)
        scalars = _nan_fill(scalars)
        if fit: scaler = StandardScaler().fit(scalars)
        sz = scaler.transform(scalars).astype(np.float32)
        X = np.concatenate([canon, sz], axis=1)
    else:
        X = canon
    y = np.array([r["label"] for r in recs], dtype=np.int64)
    return X, y, scaler


def train_one_variant(variant, keys):
    print(f"\n--- Training Variant {variant} ({len(keys)} scalar features) ---")
    X_tr, y_tr, scaler = build_X_y(train_recs, keys, fit=True)
    X_te, y_te, _      = build_X_y(test_recs,  keys, scaler=scaler)
    in_dim = X_tr.shape[1]
    print(f"  X shape: train {X_tr.shape}   test {X_te.shape}")

    model = MINDPlusClassifier(in_dim)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    loss_fn = nn.CrossEntropyLoss()
    train_loader = DataLoader(_FeatureDataset(X_tr, y_tr), batch_size=32, shuffle=True)

    history = []
    for epoch in range(10):
        model.train(); total_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad(); logits = model(X)
            loss = loss_fn(logits, y); loss.backward(); opt.step()
            total_loss += loss.item() * X.size(0)
        history.append(total_loss / len(train_loader.dataset))

    # Eval on Wikipedia test
    model.eval()
    with torch.no_grad():
        X_te_t = torch.from_numpy(X_te).to(device)
        logits = model(X_te_t).cpu().numpy()
        probs  = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
        preds  = (probs > 0.5).astype(int)
    acc = accuracy_score(y_te, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average="binary", zero_division=0)
    try: auc = float(roc_auc_score(y_te, probs)) if len(set(y_te.tolist())) > 1 else float("nan")
    except ValueError: auc = float("nan")
    try: brier = float(brier_score_loss(y_te, probs))
    except ValueError: brier = float("nan")
    cm = confusion_matrix(y_te, preds, labels=[0, 1])

    wiki_metrics = {
        "n": int(len(y_te)), "accuracy": float(acc),
        "precision": float(prec), "recall": float(rec), "f1": float(f1),
        "auc_roc": auc, "brier": brier,
        "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1])},
        "label_distribution_test": {"y=0": int((y_te==0).sum()), "y=1": int((y_te==1).sum())},
        "prob_min": round(float(probs.min()), 4), "prob_max": round(float(probs.max()), 4),
        "prob_mean": round(float(probs.mean()), 4),
    }
    print(f"  Wiki: acc={acc:.4f}  f1={f1:.4f}  AUROC={auc:.4f}  Brier={brier:.4f}")

    # Save checkpoint
    ckpt = {"model_state": model.state_dict(), "input_dim": in_dim,
            "scaler_mean": (scaler.mean_.tolist() if scaler else None),
            "scaler_scale": (scaler.scale_.tolist() if scaler else None),
            "keys": keys, "variant": variant}
    pth_path = f"{MODEL_TAG}_variant_{variant}_best.pth"
    torch.save(ckpt, pth_path)
    print(f"  \u2713 saved {pth_path}")

    return model, scaler, in_dim, keys, wiki_metrics, history


print(f"\nTraining {len(VARIANTS)} variants on OPT-6.7B hidden_dim={hidden_dim} ...")
_t0 = time.perf_counter()
for variant, keys in VARIANTS.items():
    mdl, sc, in_dim, kk, wiki_m, hist = train_one_variant(variant, keys)
    TRAINED[variant] = {"model": mdl, "scaler": sc, "in_dim": in_dim, "keys": kk}
    DIAG["variants"][variant] = {
        "feature_keys": [f"canonical(hidden_dim)"] + kk,
        "input_dim": in_dim,
        "mlp_epoch_history": hist,
        "wikipedia_eval": wiki_m,
        "multitask": {},   # filled in by Stage 6
    }
DIAG["stage_timings_sec"]["variant_training"] = round(time.perf_counter() - _t0, 2)
print(f"\n[OK] All variants trained in {DIAG['stage_timings_sec']['variant_training']}s")


In [ ]:
# =============================================================================
# BLOCK 5 (STAGE 5): LOAD 10 MULTI-TASK EVAL DATASETS (parquet -> HF fallback)
# =============================================================================
from datasets import load_dataset, Dataset as HFDataset
import pyarrow.parquet as pq

# Local parquet candidates checked in order
_LOCAL_PARQUET_CANDIDATES = [
    "eval_{label}.parquet",
    "./eval_datasets/eval_{label}.parquet",
    "/kaggle/input/dissertation-eval-datasets/eval_{label}.parquet",
    "/kaggle/input/eval-datasets/eval_{label}.parquet",
    "/kaggle/working/eval_{label}.parquet",
    "/content/drive/MyDrive/eval_datasets/eval_{label}.parquet",
    "/content/eval_{label}.parquet",
]

def _find_local_parquet(label):
    for tpl in _LOCAL_PARQUET_CANDIDATES:
        p = tpl.format(label=label)
        if os.path.exists(p):
            return p
    return None


def safe_load_first(*tries, subsample=None, label=""):
    local = _find_local_parquet(label)
    if local is not None:
        try:
            tbl = pq.read_table(local).to_pandas()
            ds = HFDataset.from_pandas(tbl)
            if subsample is not None and subsample < len(ds):
                ds = ds.select(range(subsample))
            print(f"  \u2713 {label}: {len(ds)} samples (LOCAL: {local})")
            return ds
        except Exception as e:
            print(f"  [warn] local parquet load failed for {label}: {e}")
    for i, fn in enumerate(tries, 1):
        try:
            ds = fn()
            if subsample is not None and subsample < len(ds):
                ds = ds.select(range(subsample))
            print(f"  \u2713 {label}: {len(ds)} samples (HF loader #{i})")
            return ds
        except Exception as e:
            print(f"  [warn] HF loader #{i} failed for {label}: {e}")
    return None


print("Loading 10 eval datasets ...")
_t0 = time.perf_counter()
DATASETS = {}

DATASETS["truthfulqa"] = safe_load_first(
    lambda: load_dataset("truthfulqa/truthful_qa", "generation", split="validation"),
    lambda: load_dataset("truthful_qa", "generation", split="validation"),
    label="truthfulqa")

DATASETS["triviaqa"] = safe_load_first(
    lambda: load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation"),
    lambda: load_dataset("trivia_qa", "rc.nocontext", split="validation"),
    label="triviaqa")

DATASETS["coqa"] = safe_load_first(
    lambda: load_dataset("stanfordnlp/coqa", split="validation"),
    lambda: load_dataset("coqa", split="validation"),
    subsample=500, label="coqa")

def _load_tydi():        return load_dataset("google-research-datasets/tydiqa", "secondary_task", split="validation")
def _load_tydi_legacy(): return load_dataset("tydiqa", "secondary_task", split="validation")
DATASETS["tydiqa"] = safe_load_first(_load_tydi, _load_tydi_legacy, label="tydiqa")

for label, cfg in [("halueval_qa","qa"), ("halueval_summ","summarization"), ("halueval_dialog","dialogue")]:
    DATASETS[label] = safe_load_first(
        lambda c=cfg: load_dataset("pminervini/HaluEval", c, split="data"),
        lambda c=cfg: load_dataset("HaluEval", c, split="data"),
        label=label)

DATASETS["nq_open"] = safe_load_first(
    lambda: load_dataset("google-research-datasets/nq_open", split="validation"),
    lambda: load_dataset("nq_open", split="validation"),
    label="nq_open")

DATASETS["hotpotqa"] = safe_load_first(
    lambda: load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation"),
    lambda: load_dataset("hotpot_qa", "distractor", split="validation"),
    label="hotpotqa")

DATASETS["popqa"] = safe_load_first(
    lambda: load_dataset("akariasai/PopQA", split="test"),
    lambda: load_dataset("PopQA", split="test"),
    label="popqa")

DIAG["downstream_datasets"]["loaded"] = {k: (len(v) if v is not None else 0) for k, v in DATASETS.items()}
DIAG["stage_timings_sec"]["dataset_load"] = round(time.perf_counter() - _t0, 2)
print(f"Loaded in {DIAG['stage_timings_sec']['dataset_load']}s")


In [ ]:
# =============================================================================
# BLOCK 6 (STAGE 6): MULTI-TASK EVAL — 4 NEW VARIANTS × 10 DATASETS
# =============================================================================
from tqdm.auto import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LL = need_llm()
tokenizer, model_llm = LL["tokenizer"], LL["model"]

MAX_GEN_NEW = 48
# --- QUICK-EXPERIMENT EVAL CAP (added 2026-06-02) ---
# Cap each downstream dataset to a fixed number of DETERMINISTIC first-N rows so that
# 02 (variants) and 03 (baselines) evaluate the EXACT same portion of every dataset.
# Set QUICK_EVAL_N = None to restore the original 20% (DOWNSTREAM_SCALE_CAP) scaling.
QUICK_EVAL_N = 500
DOWNSTREAM_SCALE_CAP = 0.2   # (legacy) only used when QUICK_EVAL_N is None
def _eval_cap(n):
    return min(QUICK_EVAL_N, n) if QUICK_EVAL_N is not None else max(20, int(n * DOWNSTREAM_SCALE_CAP))

@torch.no_grad()
def generate_short_answer(prompt, max_new=MAX_GEN_NEW):
    _max_pos = getattr(model_llm.config, "max_position_embeddings", 4096)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=max(_max_pos - max_new, 256)).to(model_llm.device)
    out = model_llm.generate(**enc, max_new_tokens=max_new,
                              do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip()


def score_match(gen: str, gold) -> int:
    """Return 1 if `gen` does NOT match `gold` (hallucination), else 0."""
    if isinstance(gold, str): gold = [gold]
    if not gold: return 1
    gen_norm = gen.lower().strip()
    for g in gold:
        if g and g.lower().strip() in gen_norm: return 0
        if g and gen_norm in g.lower().strip(): return 0
    return 1


def _metrics(y, p, pr):
    y = np.asarray(y); p = np.asarray(p); pr = np.asarray(pr)
    if len(y) == 0: return {"n": 0}
    acc = float(accuracy_score(y, p))
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    try: auc = float(roc_auc_score(y, pr)) if len(set(y.tolist())) > 1 else float("nan")
    except ValueError: auc = float("nan")
    try: brier = float(brier_score_loss(y, pr))
    except ValueError: brier = float("nan")
    cm = confusion_matrix(y, p, labels=[0, 1])
    return {"n": int(len(y)), "accuracy": acc, "precision": float(prec), "recall": float(rec),
            "f1": float(f1), "auc_roc": auc, "brier": brier,
            "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1])},
            "label_distribution": {"y=0": int((y==0).sum()), "y=1": int((y==1).sum())},
            "prob_min": round(float(pr.min()) if len(pr) else 0.0, 4),
            "prob_max": round(float(pr.max()) if len(pr) else 0.0, 4),
            "prob_mean": round(float(pr.mean()) if len(pr) else 0.0, 4)}


def _features_for_text(text):
    """Run the LLM ONCE on `text`; return the (variant-independent) NEW-feature dict
    with the F11 URP vector flattened into F11_urp_<j> scalar keys."""
    feats = extract_all_new_features(text)
    for j, v in enumerate(feats["F11_urp"]):
        feats[f"F11_urp_{j}"] = float(v)
    return feats


def _classify_feats(feats, variant):
    """Classify ALREADY-EXTRACTED features with TRAINED[variant]. No LLM call here, so the
    same `feats` can be scored by every variant — each variant just selects its own keys."""
    t = TRAINED[variant]
    canon = np.array(feats["embedding"], dtype=np.float32).reshape(1, -1)
    keys = t["keys"]
    if keys:
        scalars = np.array([[feats.get(k, 0.0) for k in keys]], dtype=np.float32)
        scalars = _nan_fill(scalars)
        if t["scaler"] is not None:
            scalars = t["scaler"].transform(scalars).astype(np.float32)
        X = np.concatenate([canon, scalars], axis=1)
    else:
        X = canon
    with torch.no_grad():
        Xt = torch.from_numpy(X).to(device)
        logits = t["model"](Xt)
        prob = float(torch.softmax(logits, dim=-1)[0, 1].cpu().item())
    return int(prob > 0.5), prob


def classify_with_variant(text, variant):
    """Backward-compatible single-variant wrapper (extract + classify)."""
    return _classify_feats(_features_for_text(text), variant)


def eval_open(ds, prompt_fn, gold_fn, dsname, variants):
    """Generate ONCE per sample, extract features ONCE, then score that same text with
    EVERY variant. Returns {variant: metrics}. Identical results to the old per-variant
    loop, minus the redundant re-generation."""
    cap = _eval_cap(len(ds))
    selected = ds.select(range(min(cap, len(ds))))
    acc = {v: {"y": [], "p": [], "pr": []} for v in variants}
    fails = 0
    for s in tqdm(selected, desc=f"{dsname}"):
        try:
            prompt = prompt_fn(s)
            gen = generate_short_answer(prompt)
            label = score_match(gen, gold_fn(s))
            feats = _features_for_text((prompt + " " + gen).strip())
            for v in variants:
                pp, pr = _classify_feats(feats, v)
                acc[v]["y"].append(label); acc[v]["p"].append(pp); acc[v]["pr"].append(pr)
        except Exception:
            fails += 1; continue
    return {v: _metrics(d["y"], d["p"], d["pr"]) for v, d in acc.items()}, fails


def eval_he(ds, prompt_fn, right_key, wrong_key, dsname, variants):
    """HaluEval: no generation. Extract features ONCE per (right/wrong) answer text, then
    score with EVERY variant. Returns {variant: metrics}."""
    cap = _eval_cap(len(ds))
    selected = ds.select(range(min(cap, len(ds))))
    acc = {v: {"y": [], "p": [], "pr": []} for v in variants}
    fails = 0
    for s in tqdm(selected, desc=f"{dsname}"):
        try:
            prompt = prompt_fn(s)
            for ak, gl in [(right_key, 0), (wrong_key, 1)]:
                feats = _features_for_text((prompt + " " + s[ak]).strip())
                for v in variants:
                    pp, pr = _classify_feats(feats, v)
                    acc[v]["y"].append(gl); acc[v]["p"].append(pp); acc[v]["pr"].append(pr)
        except Exception:
            fails += 1; continue
    return {v: _metrics(d["y"], d["p"], d["pr"]) for v, d in acc.items()}, fails


# Build EVAL_TASKS
EVAL_TASKS = []
if DATASETS.get("truthfulqa") is not None:
    EVAL_TASKS.append(("truthfulqa", "open", DATASETS["truthfulqa"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        lambda s: s.get("correct_answers", []), None, None))
if DATASETS.get("triviaqa") is not None:
    EVAL_TASKS.append(("triviaqa", "open", DATASETS["triviaqa"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        lambda s: list(s["answer"].get("aliases", [])) + [s["answer"].get("value", "")],
        None, None))
if DATASETS.get("coqa") is not None:
    def _cpr(s):
        ctx = s["story"][:600]
        q = s["questions"][0] if s["questions"] else ""
        return f"Context: {ctx} Q: {q} A:"
    def _cg(s):
        a = s["answers"]; return a["input_text"][0] if a["input_text"] else ""
    EVAL_TASKS.append(("coqa", "open", DATASETS["coqa"], _cpr, _cg, None, None))
if DATASETS.get("tydiqa") is not None:
    EVAL_TASKS.append(("tydiqa", "open", DATASETS["tydiqa"],
        lambda s: f"Context: {s['context'][:600]} Q: {s['question']} A:",
        lambda s: s.get("answers", {}).get("text", []), None, None))
if DATASETS.get("halueval_qa") is not None:
    EVAL_TASKS.append(("halueval_qa", "he", DATASETS["halueval_qa"],
        lambda s: f"Context: {s['knowledge'][:400]} Q: {s['question']} A:",
        None, "right_answer", "hallucinated_answer"))
if DATASETS.get("halueval_summ") is not None:
    EVAL_TASKS.append(("halueval_summ", "he", DATASETS["halueval_summ"],
        lambda s: f"{s['document'][:500]} Summary:", None, "right_summary", "hallucinated_summary"))
if DATASETS.get("halueval_dialog") is not None:
    EVAL_TASKS.append(("halueval_dialog", "he", DATASETS["halueval_dialog"],
        lambda s: f"Knowledge: {s['knowledge'][:300]}\nDialogue: {s['dialogue_history'][:300]}\n[Assistant]:",
        None, "right_response", "hallucinated_response"))
if DATASETS.get("nq_open") is not None:
    EVAL_TASKS.append(("nq_open", "open", DATASETS["nq_open"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        lambda s: s.get("answer", []) if isinstance(s.get("answer"), list) else [s.get("answer", "")],
        None, None))
if DATASETS.get("hotpotqa") is not None:
    def _hotpot_pr(s):
        try:
            sents = s["context"]["sentences"]
            ctx = " ".join([" ".join(p) for p in sents[:3]])[:600]
        except Exception: ctx = ""
        return f"Context: {ctx} Q: {s['question']} A:"
    EVAL_TASKS.append(("hotpotqa", "open", DATASETS["hotpotqa"], _hotpot_pr,
        lambda s: [s.get("answer", "")], None, None))
if DATASETS.get("popqa") is not None:
    import json as _json_popqa
    def _popqa_gold(s):
        pa = s.get("possible_answers", "[]")
        if isinstance(pa, str):
            try: pa = _json_popqa.loads(pa)
            except Exception: pa = [pa]
        return pa if isinstance(pa, list) else [pa]
    EVAL_TASKS.append(("popqa", "open", DATASETS["popqa"],
        lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        _popqa_gold, None, None))

print(f"\nMulti-task eval — {len(VARIANTS)} variants × {len(EVAL_TASKS)} datasets ...")
_t0 = time.perf_counter()
fails_per = {}
for task in EVAL_TASKS:
    ds_name, ds_kind, ds, prompt_fn, gold_fn, rk, wk = task
    print(f"\n--- {ds_name} ({ds_kind}, n={len(ds)}) ---")
    fails_per[ds_name] = {}
    _variant_list = list(VARIANTS.keys())
    if ds_kind == "open":
        res, f = eval_open(ds, prompt_fn, gold_fn, ds_name, _variant_list)
    else:
        res, f = eval_he(ds, prompt_fn, rk, wk, ds_name, _variant_list)
    for variant in _variant_list:
        m = res[variant]
        DIAG["variants"][variant]["multitask"][ds_name] = m
        fails_per[ds_name][variant] = int(f)
        auc = m.get("auc_roc"); auc_s = f"{auc:.3f}" if isinstance(auc, float) and auc == auc else "--"
        print(f"    Variant {variant}: AUROC={auc_s}  n={m.get('n', 0)}  fails={f}")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
DIAG["downstream_datasets"]["n_eval_failures_per_dataset"] = fails_per
DIAG["stage_timings_sec"]["multitask_eval"] = round(time.perf_counter() - _t0, 2)


In [ ]:
# =============================================================================
# BLOCK 7 (STAGE 7): CONSOLIDATED DUMP + AUTO-DOWNLOAD
# =============================================================================
DIAG["timestamp_utc"] = (datetime.datetime.now(datetime.timezone.utc)
                          .replace(microsecond=0).isoformat().replace("+00:00", "Z"))
DIAG["host"] = ("kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ
                else ("colab" if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
                      else "local"))
DIAG["stage_timings_sec"]["total"] = round(sum(DIAG["stage_timings_sec"].values()), 2)

with open(PATH_RESULTS, "w") as f:
    json.dump(DIAG, f, indent=2, default=str)
print(f"\n\u2713 wrote {PATH_RESULTS}  ({os.path.getsize(PATH_RESULTS)/1024:.1f} KB)")

# Summary
print("\n" + "=" * 88)
print(f"NEW-FEATURE VARIANTS — OPT-6.7B")
print("=" * 88)
ds_order = ["truthfulqa","triviaqa","coqa","tydiqa","halueval_qa","halueval_summ","halueval_dialog","nq_open","hotpotqa","popqa"]
ds_short = {"truthfulqa":"TruthQA","triviaqa":"TrivQA","coqa":"CoQA","tydiqa":"TydiQA",
             "halueval_qa":"HE-QA","halueval_summ":"HE-Sum","halueval_dialog":"HE-Dial",
             "nq_open":"NQ","hotpotqa":"Hotpot","popqa":"PopQA"}
hdr = f"{'Variant':10s} {'Wiki':>7s} "
for ds in ds_order: hdr += f"{ds_short[ds]:>8s} "
hdr += f"{'avg':>8s}"
print(hdr); print("-" * 88)
for v in ["M", "N", "O", "P"]:
    if v not in DIAG["variants"]: continue
    info = DIAG["variants"][v]
    wiki_auc = info.get("wikipedia_eval", {}).get("auc_roc", float("nan"))
    row = f"{v:10s} {wiki_auc:>7.3f} "
    aucs = []
    for ds in ds_order:
        m = info["multitask"].get(ds, {})
        auc = m.get("auc_roc")
        if auc is not None and auc == auc:
            aucs.append(auc); row += f"{auc:>8.3f} "
        else:
            row += f"{'--':>8s} "
    if aucs: row += f"{sum(aucs)/len(aucs):>8.3f}"
    else:    row += f"{'--':>8s}"
    print(row)
print("=" * 88)
print(f"\nTimings:")
for k, v in DIAG["stage_timings_sec"].items():
    print(f"  {k:30s} {v:>8.2f} s")
print(f"\nPaste {PATH_RESULTS} back to the assistant for cross-method comparison "
      f"against the existing OPT-6.7B baselines + variants A-L.")


# Auto-download
def _auto_download(file_patterns):
    import os, glob
    resolved = []
    for fp in file_patterns:
        if "*" in fp or "?" in fp:
            resolved.extend(sorted(glob.glob(fp)))
        elif os.path.exists(fp):
            resolved.append(fp)
    print("\nFINAL OUTPUTS")
    print("-" * 60)
    for f in resolved:
        print(f"  {f}   ({os.path.getsize(f)/1e6:.2f} MB)")
    if not resolved: return
    try:
        from google.colab import files as _colab_files
        print("\nColab detected -> triggering browser downloads ...")
        for f in resolved:
            try:    _colab_files.download(f)
            except Exception as e: print(f"  download failed for {f}: {e}")
        return
    except ImportError:
        pass
    if os.path.exists("/kaggle/working"):
        print("\nKaggle detected -> files persist in /kaggle/working/. Download via 'Output' panel.")
        return
    print(f"\nLocal Jupyter -> files are in {os.getcwd()}")


_auto_download([
    PATH_RESULTS,
    f"{MODEL_TAG}_variant_M_best.pth",
    f"{MODEL_TAG}_variant_N_best.pth",
    f"{MODEL_TAG}_variant_O_best.pth",
    f"{MODEL_TAG}_variant_P_best.pth",
])
